# Kaggle Cardiovascular Disease Dataset - Preprocessing & Training
**Dataset**: 70,000 patients for cardiovascular disease prediction
**Author**: AI Assistant
**Date**: 2026-04-04

## 1. Setup & Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# XGBoost
from xgboost import XGBClassifier

# Save models
import pickle
import os
import json

print('All libraries imported successfully!')

## 2. Load Dataset

In [ ]:
# Path to Kaggle dataset
DATA_PATH = r'd:\UIT\DU AN TOT NGHIEP\heart-disease-diagnosis\data\raw\cardio_train.csv'

# Load data (semicolon separator)
df = pd.read_csv(DATA_PATH, sep=';')

print(f'Dataset shape: {df.shape}')
print(f'\nFirst 5 rows:')
df.head()

## 3. Initial Data Analysis

In [ ]:
print('='*70)
print('INITIAL DATA ANALYSIS')
print('='*70)

print(f'\nShape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nData types:\n{df.dtypes}')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nTarget distribution:\n{df["cardio"].value_counts()}')

## 4. Convert Age from Days to Years

In [ ]:
# Convert age from days to years
df['age_years'] = df['age'] / 365.25

print(f'Age in years - Min: {df["age_years"].min():.1f}, Max: {df["age_years"].max():.1f}, Mean: {df["age_years"].mean():.1f}')

# Drop original 'age' column (in days) and 'id' column
df = df.drop(['age', 'id'], axis=1)

print(f'\nAfter dropping age (days) and id: {df.shape}')

## 5. Data Cleaning - Outliers Removal

In [ ]:
print('='*70)
print('DATA CLEANING - OUTLIERS')
print('='*70)

df_original = df.copy()
print(f'Original rows: {len(df)}')

# Define valid ranges
valid_ranges = {
    'height': (100, 220),      # cm
    'weight': (30, 200),      # kg
    'ap_hi': (80, 250),       # systolic BP
    'ap_lo': (50, 150),       # diastolic BP
    'age_years': (20, 80)     # years
}

# Apply filters
for col, (min_val, max_val) in valid_ranges.items():
    before = len(df)
    df = df[(df[col] >= min_val) & (df[col] <= max_val)]
    removed = before - len(df)
    if removed > 0:
        print(f'  {col}: removed {removed} rows (range {min_val}-{max_val})')

# Remove impossible: ap_lo > ap_hi
impossible = (df['ap_lo'] > df['ap_hi']).sum()
df = df[df['ap_lo'] <= df['ap_hi']]
print(f'  ap_lo > ap_hi: removed {impossible} rows')

print(f'\nAfter cleaning: {len(df)} rows (removed {len(df_original) - len(df)} rows, {100*(len(df_original) - len(df))/len(df_original):.1f}%)')

## 6. Feature Engineering

In [ ]:
# BMI
df['bmi'] = df['weight'] / ((df['height']/100) ** 2)

# Pulse pressure (difference between systolic and diastolic)
df['pulse_pressure'] = df['ap_hi'] - df['ap_lo']

# Mean Arterial Pressure (MAP)
df['map'] = (df['ap_hi'] + 2 * df['ap_lo']) / 3

# Age groups
df['age_group'] = pd.cut(df['age_years'], bins=[0, 40, 50, 55, 60, 100], labels=[0, 1, 2, 3, 4])
df['age_group'] = df['age_group'].astype(int)

# BMI categories
df['bmi_category'] = pd.cut(df['bmi'], bins=[0, 18.5, 25, 30, 100], labels=[0, 1, 2, 3])
df['bmi_category'] = df['bmi_category'].astype(int)

# Blood pressure categories
df['bp_category'] = pd.cut(df['ap_hi'], bins=[0, 120, 130, 140, 200], labels=[0, 1, 2, 3])
df['bp_category'] = df['bp_category'].astype(int)

print('New features created:')
print('  - bmi: Body Mass Index')
print('  - pulse_pressure: Systolic - Diastolic')
print('  - map: Mean Arterial Pressure')
print('  - age_group: 0=<40, 1=40-50, 2=50-55, 3=55-60, 4=60+')
print('  - bmi_category: 0=underweight, 1=normal, 2=overweight, 3=obese')
print('  - bp_category: 0=normal, 1=elevated, 2=high, 3=very high')

print(f'\nFinal feature set: {df.columns.tolist()}')

## 7. Prepare Data for Training

In [ ]:
# Separate features and target
X = df.drop('cardio', axis=1)
y = df['cardio']

# Split train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Split train into train/val (75/25 of train)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)

print(f'Train set: {X_train.shape[0]} samples')
print(f'Validation set: {X_val.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'\nFeatures ({len(X.columns)}): {X.columns.tolist()}')

## 8. Scale Features

In [ ]:
# Scale numerical features
scaler = StandardScaler()

# Fit on train, transform train/val/test
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print('Features scaled using StandardScaler')
print(f'Scaled shape: {X_train_scaled.shape}')

## 9. Train Models

In [ ]:
# Define models to train
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, eval_metric='logloss', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

# Train and evaluate each model
results = {}

for name, model in models.items():
    print(f'Training {name}...', end=' ')
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results[name] = {
        'model': model,
        'accuracy': acc,
        'roc_auc': auc
    }
    
    print(f'Acc: {acc:.4f}, AUC: {auc:.4f}')

print('\n' + '='*70)
print('MODEL COMPARISON')
print('='*70)

In [ ]:
# Sort by AUC
sorted_results = sorted(results.items(), key=lambda x: x[1]['roc_auc'], reverse=True)

print(f'\n{"Model":<25} {"Accuracy":>10} {"ROC-AUC":>10}')
print('-'*45)
for name, res in sorted_results:
    print(f'{name:<25} {res["accuracy"]:>10.4f} {res["roc_auc"]:>10.4f}')

## 10. Save Models

In [ ]:
# Create output directory
MODEL_DIR = r'd:\UIT\DU AN TOT NGHIEP\heart-disease-diagnosis\models\saved_models\kaggle_v1'
os.makedirs(MODEL_DIR, exist_ok=True)

# Save scaler
scaler_path = os.path.join(MODEL_DIR, 'scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f'Saved scaler: {scaler_path}')

# Save each model
for name, res in results.items():
    filename = name.lower().replace(' ', '_') + '_model.pkl'
    filepath = os.path.join(MODEL_DIR, filename)
    
    with open(filepath, 'wb') as f:
        pickle.dump(res['model'], f)
    
    print(f'Saved {name}: {filepath}')

# Save summary JSON
summary = {}
for name, res in results.items():
    key = name.lower().replace(' ', '_')
    summary[key] = {
        'accuracy': float(res['accuracy']),
        'roc_auc': float(res['roc_auc']),
        'model_type': key
    }

summary_path = os.path.join(MODEL_DIR, 'models_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\nSaved summary: {summary_path}')

## 11. Save Processed Data

In [ ]:
# Save processed datasets
DATA_PROCESSED_DIR = r'd:\UIT\DU AN TOT NGHIEP\heart-disease-diagnosis\data\processed'
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)

# Combine X and y for saving
train_df = pd.concat([X_train, y_train], axis=1)
val_df = pd.concat([X_val, y_val], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

# Save to CSV
train_df.to_csv(os.path.join(DATA_PROCESSED_DIR, 'kaggle_train.csv'), index=False)
val_df.to_csv(os.path.join(DATA_PROCESSED_DIR, 'kaggle_val.csv'), index=False)
test_df.to_csv(os.path.join(DATA_PROCESSED_DIR, 'kaggle_test.csv'), index=False)

print(f'Saved processed datasets to {DATA_PROCESSED_DIR}:')
print(f'  - kaggle_train.csv: {train_df.shape[0]} rows')
print(f'  - kaggle_val.csv: {val_df.shape[0]} rows')
print(f'  - kaggle_test.csv: {test_df.shape[0]} rows')

## 12. Summary

In [ ]:
print('='*70)
print('PREPROCESSING & TRAINING COMPLETE')
print('='*70)
print(f'\nOriginal dataset: 70,000 rows')
print(f'After cleaning: {len(df)} rows')
print(f'\nFeatures used ({len(X.columns)}):')
for col in X.columns:
    print(f'  - {col}')
print(f'\nBest model: {sorted_results[0][0]} (AUC: {sorted_results[0][1]["roc_auc"]:.4f})')
print(f'\nOutput directory: {MODEL_DIR}')